# Notebook d'extraction - Groupe D : fraude à l'ouverture de compte

**MBA 2 Data & BI — Epitech — Atelier fil rouge, module 1**

Ce notebook télécharge `Base.csv` (Bank Account Fraud Dataset Suite, Feedzai, NeurIPS 2022), contrôle sa qualité, le nettoie au bon niveau de granularité (**1 ligne = 1 demande d'ouverture de compte**) et produit :

| Fichier produit | Contenu |
|---|---|
| `data/processed/base_clean.parquet` | Jeu propre, prêt pour l'EDA du module 2 |
| `data/processed/dictionnaire_variables.csv` | Dictionnaire des variables (type, disponibilité au dépôt, statut, manquants) |
| `data/processed/rapport_qualite.json` | Résultats des contrôles qualité sur le fichier brut |
| `lineage.json` | Journal de lineage : source, URL, date, version, SHA-256, transformations |

## Lancer le notebook

Pour exécuter le notebook : Placez le fichier Base.csv téléchargé depuis Kaggle dans le dossier data/raw/ puis lancez Kernel > Restart & Run All.

In [43]:
import hashlib
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

# --- Source & Informations ---
SOURCE_URL = "https://www.kaggle.com/datasets/sgpjesus/bank-account-fraud-dataset-neurips2022"
DOC_URL = "https://github.com/feedzai/bank-account-fraud"
DATASHEET_URL = "https://github.com/feedzai/bank-account-fraud"
FILE_NAME = "Base.csv"
LICENCE = "CC BY-NC-SA 4.0" 
KAGGLE_HANDLE = "sgpjesus/bank-account-fraud-dataset-neurips2022"

# --- Chemins ---
ROOT = Path.cwd()

if (ROOT / "Base.csv").exists():
    RAW_FILE = ROOT / "Base.csv"
elif (ROOT / "raw" / "Base.csv").exists():
    RAW_FILE = ROOT / "raw" / "Base.csv"
else:
    RAW_FILE = ROOT / "data" / "raw" / "Base.csv"

RAW_DIR = RAW_FILE.parent
OUT_DIR = RAW_DIR.parent / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Paramètres métier ---
TARGET = "fraud_bool"
TRAIN_MONTHS = list(range(0, 6))   # Mois 0 à 5 : Entraînement
TEST_MONTHS = list(range(6, 8))    # Mois 6 et 7 : Test
EXPECTED_ROWS = 1_000_000

print(f"Fichier source : {RAW_FILE}")
print(f"Python {sys.version.split()[0]} | Pandas {pd.__version__}")

Fichier source : /Users/sandrine/Desktop/data/raw/Base.csv
Python 3.12.13 | Pandas 3.0.6


## 1. Fonctions utilitaires

In [30]:
def sha256(path: Path, chunk_size: int = 1 << 20) -> str:
    # Empreinte SHA-256 d'un fichier, lue par blocs pour ne pas saturer la mémoire.
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk_size), b""):
            h.update(block)
    return h.hexdigest()


TRANSFORMATIONS = []

def log_step(name: str, description: str, before: pd.DataFrame, after: pd.DataFrame, **details):
    # Enregistre une transformation dans le journal de lineage.
    step = {
        "etape": len(TRANSFORMATIONS) + 1,
        "nom": name,
        "description": description,
        "lignes_avant": int(before.shape[0]),
        "lignes_apres": int(after.shape[0]),
        "colonnes_avant": int(before.shape[1]),
        "colonnes_apres": int(after.shape[1]),
    }
    step.update(details)
    TRANSFORMATIONS.append(step)
    print(f"[{step['etape']}] {name} : {before.shape} -> {after.shape}")

## 2. Téléchargement

In [31]:
# 2. Vérification du fichier local et génération des métadonnées
if not RAW_FILE.exists():
    raise FileNotFoundError(f"Fichier introuvable ! Assurez-vous que Base.csv est bien placé dans {RAW_FILE}")

raw_sha256 = sha256(RAW_FILE)
raw_size = RAW_FILE.stat().st_size

download_meta = {
    "methode": "Téléchargement manuel depuis Kaggle",
    "version": "1",
    "date_recuperation": datetime.now(timezone.utc).isoformat(timespec="seconds")
}

print(f"Fichier Base.csv détecté avec succès : {RAW_FILE}")
print(f"SHA-256 : {raw_sha256}")
print(f"Taille  : {raw_size / 1e6:.1f} Mo")

Fichier Base.csv détecté avec succès : /Users/sandrine/Desktop/data/raw/Base.csv
SHA-256 : 7bf10a37ce07e72e14c1b09e5efee3d27261baff4facc7da767b0474dcf9b809
Taille  : 213.4 Mo


## 3. Chargement et contrôle du schéma

In [32]:
EXPECTED_COLUMNS = [
    "fraud_bool", "income", "name_email_similarity", "prev_address_months_count",
    "current_address_months_count", "customer_age", "days_since_request",
    "intended_balcon_amount", "payment_type", "zip_count_4w", "velocity_6h",
    "velocity_24h", "velocity_4w", "bank_branch_count_8w",
    "date_of_birth_distinct_emails_4w", "employment_status", "credit_risk_score",
    "email_is_free", "housing_status", "phone_home_valid", "phone_mobile_valid",
    "bank_months_count", "has_other_cards", "proposed_credit_limit", "foreign_request",
    "source", "session_length_in_minutes", "device_os", "keep_alive_session",
    "device_distinct_emails_8w", "device_fraud_count", "month",
]

df_raw = pd.read_csv(RAW_FILE)
print("Dimensions brutes :", df_raw.shape)

# Une éventuelle colonne d'index exportée (« Unnamed: 0 ») n'est pas une variable.
df = df_raw.copy()
index_cols = [c for c in df.columns if str(c).startswith("Unnamed")]
if index_cols:
    before = df
    df = df.drop(columns=index_cols)
    log_step("suppression_index", f"Suppression de la colonne d'index exportée {index_cols}", before, df)

missing_cols = sorted(set(EXPECTED_COLUMNS) - set(df.columns))
extra_cols = sorted(set(df.columns) - set(EXPECTED_COLUMNS))
assert not missing_cols, f"Colonnes attendues absentes : {missing_cols}"
assert not extra_cols, f"Colonnes inattendues : {extra_cols}"
df = df[EXPECTED_COLUMNS]
print("Schéma conforme : 32 colonnes attendues présentes.")
df.head()

Dimensions brutes : (1000000, 32)
Schéma conforme : 32 colonnes attendues présentes.


,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,...,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
0,0,0.3,0.986506,-1,25,40,0.006735,102.453711,AA,1059,...,0,1500.0,0,INTERNET,16.224843,linux,1,1,0,0
1,0,0.8,0.617426,-1,89,20,0.010095,-0.849551,AD,1658,...,0,1500.0,0,INTERNET,3.363854,other,1,1,0,0
2,0,0.8,0.996707,9,14,40,0.012316,-1.490386,AB,1095,...,0,200.0,0,INTERNET,22.730559,windows,0,1,0,0
3,0,0.6,0.475100,11,14,30,0.006991,-1.863101,AB,3483,...,0,200.0,0,INTERNET,15.215816,linux,1,1,0,0
4,0,0.9,0.842307,-1,29,40,5.742626,47.152498,AA,2339,...,0,200.0,0,INTERNET,3.743048,other,0,1,0,0


## 4. Contrôles qualité sur le fichier brut

Les contrôles sont calculés sur les données, pas supposés : la liste des colonnes contenant des -1 sert à vérifier la documentation.

In [33]:
quality = {}

# Volume et granularité
quality["nb_lignes"] = int(len(df))
quality["nb_colonnes"] = int(df.shape[1])
if len(df) != EXPECTED_ROWS:
    print(f"ATTENTION : {len(df):,} lignes au lieu de {EXPECTED_ROWS:,} attendues.")

# Cible
target_values = sorted(df[TARGET].unique().tolist())
assert target_values == [0, 1], f"Cible non binaire : {target_values}"
quality["nb_fraudes"] = int(df[TARGET].sum())
quality["taux_fraude_pct"] = round(100 * df[TARGET].mean(), 3)

# Doublons (pas d'identifiant : seuls les doublons exacts sont détectables)
quality["doublons_exacts"] = int(df.duplicated().sum())

# Valeurs manquantes explicites et codées
quality["nan_par_colonne"] = {c: int(n) for c, n in df.isna().sum().items() if n > 0}
numeric_cols = df.select_dtypes("number").columns
quality["valeurs_moins_un"] = {c: int((df[c] == -1).sum()) for c in numeric_cols if (df[c] == -1).any()}
quality["valeurs_negatives"] = {c: int((df[c] < 0).sum()) for c in numeric_cols if (df[c] < 0).any()}

# Colonnes constantes
quality["colonnes_constantes"] = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]

# Répartition temporelle
by_month = df.groupby("month")[TARGET].agg(nb_demandes="size", nb_fraudes="sum", taux_fraude="mean")
by_month["taux_fraude"] = (100 * by_month["taux_fraude"]).round(3)
quality["par_mois"] = by_month.reset_index().to_dict(orient="records")

print(f"Lignes : {quality['nb_lignes']:,} | fraudes : {quality['nb_fraudes']:,} ({quality['taux_fraude_pct']} %)")
print("Doublons exacts :", quality["doublons_exacts"])
print("NaN explicites :", quality["nan_par_colonne"] or "aucun")
print("Valeurs -1 :", json.dumps(quality["valeurs_moins_un"], indent=2))
print("Valeurs négatives :", json.dumps(quality["valeurs_negatives"], indent=2))
print("Colonnes constantes :", quality["colonnes_constantes"] or "aucune")
by_month

Lignes : 1,000,000 | fraudes : 11,029 (1.103 %)
Doublons exacts : 0
NaN explicites : aucun
Valeurs -1 : {
  "prev_address_months_count": 712920,
  "current_address_months_count": 4254,
  "credit_risk_score": 488,
  "bank_months_count": 253635,
  "session_length_in_minutes": 2015,
  "device_distinct_emails_8w": 359
}
Valeurs négatives : {
  "prev_address_months_count": 712920,
  "current_address_months_count": 4254,
  "intended_balcon_amount": 742523,
  "velocity_6h": 44,
  "credit_risk_score": 14445,
  "bank_months_count": 253635,
  "session_length_in_minutes": 2015,
  "device_distinct_emails_8w": 359
}
Colonnes constantes : ['device_fraud_count']


,nb_demandes,nb_fraudes,taux_fraude
month,,,
0,132440,1500,1.133
1,127620,1198,0.939
2,136979,1198,0.875
3,150936,1392,0.922
4,127691,1452,1.137
5,119323,1411,1.183
6,108168,1450,1.341
7,96843,1428,1.475


In [34]:
# Modalités des variables qualitatives
for c in ["payment_type", "employment_status", "housing_status", "source", "device_os", "customer_age"]:
    print(c, "->", df[c].value_counts().sort_index().to_dict())

payment_type -> {'AA': 258249, 'AB': 370554, 'AC': 252071, 'AD': 118837, 'AE': 289}
employment_status -> {'CA': 730252, 'CB': 138288, 'CC': 37758, 'CD': 26522, 'CE': 22693, 'CF': 44034, 'CG': 453}
housing_status -> {'BA': 169675, 'BB': 260965, 'BC': 372143, 'BD': 26161, 'BE': 169135, 'BF': 1669, 'BG': 252}
source -> {'INTERNET': 992952, 'TELEAPP': 7048}
device_os -> {'linux': 332712, 'macintosh': 53826, 'other': 342728, 'windows': 263506, 'x11': 7228}
customer_age -> {10: 20987, 20: 245855, 30: 311433, 40: 238712, 50: 140353, 60: 34770, 70: 6517, 80: 1297, 90: 76}


## 5. Transformations

Chaque étape est tracée dans `TRANSFORMATIONS`, puis écrite dans `lineage.json`.

Aucune variable « à discuter » (`proposed_credit_limit`, `credit_risk_score`) n'est supprimée ici : la décision sera prise après lecture de la datasheet, et testée en M2.

In [35]:
# 5.1 Doublons exacts
before = df
df = df.drop_duplicates().reset_index(drop=True)
log_step("doublons", "Suppression des lignes entièrement identiques (pas d'identifiant disponible)", before, df)

[1] doublons : (1000000, 32) -> (1000000, 32)


In [36]:
# 5.2 Valeurs manquantes codées -> NaN + indicateur (le fait d'être manquant peut être un signal de fraude)
MISSING_AS_MINUS_ONE = [
    "prev_address_months_count", "current_address_months_count", "bank_months_count",
    "session_length_in_minutes", "device_distinct_emails_8w",
]
MISSING_AS_NEGATIVE = ["intended_balcon_amount"]

before = df.copy()
recoded = {}
for c in MISSING_AS_MINUS_ONE + MISSING_AS_NEGATIVE:
    mask = (df[c] < 0) if c in MISSING_AS_NEGATIVE else (df[c] == -1)
    if mask.any():
        df[f"{c}_manquant"] = mask.astype("int8")
        df.loc[mask, c] = np.nan
        recoded[c] = int(mask.sum())

log_step(
    "recodage_manquants",
    "-1 (ou valeurs négatives pour intended_balcon_amount) remplacés par NaN, avec une colonne indicatrice <col>_manquant",
    before, df, valeurs_recodees=recoded,
)

# Contrôle : une colonne -1 non prévue doit être signalée
# credit_risk_score peut être négatif par construction (score) : un -1 y est une vraie valeur.
SCORE_COLS = ["credit_risk_score"]
unexpected = sorted(set(quality["valeurs_moins_un"]) - set(MISSING_AS_MINUS_ONE) - set(MISSING_AS_NEGATIVE) - set(SCORE_COLS))
if unexpected:
    print("À vérifier dans la datasheet : -1 présents dans", unexpected)

[2] recodage_manquants : (1000000, 32) -> (1000000, 38)


In [37]:
# 5.3 Colonnes constantes (aucune information)
const_cols = [c for c in df.columns if c != TARGET and df[c].nunique(dropna=False) <= 1]
before = df
df = df.drop(columns=const_cols)
log_step("colonnes_constantes", f"Suppression des colonnes constantes : {const_cols or 'aucune'}", before, df,
         colonnes_supprimees=const_cols)

[3] colonnes_constantes : (1000000, 38) -> (1000000, 37)


In [38]:
# 5.4 Typage : les codes qualitatifs ne sont pas des quantités
NOMINAL = ["payment_type", "employment_status", "housing_status", "source", "device_os"]
BINARY = ["fraud_bool", "email_is_free", "phone_home_valid", "phone_mobile_valid",
          "has_other_cards", "foreign_request", "keep_alive_session"]

before = df.copy()
for c in NOMINAL:
    df[c] = df[c].astype("category")
for c in BINARY:
    df[c] = df[c].astype("int8")
df["month"] = df["month"].astype("int8")
log_step("typage", "Variables nominales en category, binaires en int8 ; customer_age (tranches) laissée numérique mais documentée comme ordinale",
         before, df)

[4] typage : (1000000, 37) -> (1000000, 37)


In [39]:
# 5.5 Découpage temporel (préparé, pas appliqué : le jeu reste complet)
before = df.copy()
df["split"] = np.where(df["month"].isin(TRAIN_MONTHS), "train", "test")
df["split"] = df["split"].astype("category")
log_step("decoupage_temporel", f"Ajout de la colonne split : train = mois {TRAIN_MONTHS}, test = mois {TEST_MONTHS}", before, df)

split_summary = df.groupby("split", observed=True)[TARGET].agg(nb_demandes="size", taux_fraude="mean")
split_summary["taux_fraude"] = (100 * split_summary["taux_fraude"]).round(3)
split_summary

[5] decoupage_temporel : (1000000, 37) -> (1000000, 38)


,nb_demandes,taux_fraude
split,,
test,205011,1.404
train,794989,1.025


## 6. Contrôles après transformation

In [40]:
assert df[TARGET].isin([0, 1]).all(), "Cible invalide"
assert df.duplicated().sum() == 0, "Doublons restants"
for c in MISSING_AS_MINUS_ONE:
    assert not (df[c] == -1).any(), f"-1 restants dans {c}"
for c in MISSING_AS_NEGATIVE:
    assert not (df[c] < 0).any(), f"Valeurs négatives restantes dans {c}"
assert set(df["split"].unique()) <= {"train", "test"}
assert len(df) == quality["nb_lignes"] - quality["doublons_exacts"], "Perte de lignes inexpliquée"
print("Tous les contrôles sont passés. Dimensions finales :", df.shape)

Tous les contrôles sont passés. Dimensions finales : (1000000, 38)


## 7. Dictionnaire des variables

In [41]:
DICTIONNAIRE = {
    # variable: (description, type, connue au dépôt, statut)
    "fraud_bool": ("Demande frauduleuse (1) ou légitime (0)", "Binaire", "Non (cible)", "Cible"),
    "income": ("Revenu déclaré, en déciles (0,1 à 0,9)", "Quantitative ordonnée", "Oui", "Garder — sensible"),
    "name_email_similarity": ("Similarité entre le nom et l'email (0 à 1)", "Quantitative continue", "Oui", "Garder"),
    "prev_address_months_count": ("Ancienneté à l'adresse précédente (mois)", "Quantitative discrète", "Oui", "Garder"),
    "current_address_months_count": ("Ancienneté à l'adresse actuelle (mois)", "Quantitative discrète", "Oui", "Garder"),
    "customer_age": ("Âge par tranches de 10 ans", "Qualitative ordinale", "Oui", "Garder — sensible"),
    "days_since_request": ("Délai depuis la création de la demande (jours)", "Quantitative continue", "Oui", "Garder"),
    "intended_balcon_amount": ("Montant initial prévu", "Quantitative continue", "Oui", "Garder"),
    "payment_type": ("Type de plan de paiement (anonymisé)", "Qualitative nominale", "Oui", "Garder"),
    "zip_count_4w": ("Demandes avec le même code postal sur 4 semaines", "Quantitative discrète", "Oui", "Garder"),
    "velocity_6h": ("Rythme moyen de demandes sur 6 h", "Quantitative continue", "Oui", "Garder"),
    "velocity_24h": ("Rythme moyen de demandes sur 24 h", "Quantitative continue", "Oui", "Garder"),
    "velocity_4w": ("Rythme moyen de demandes sur 4 semaines", "Quantitative continue", "Oui", "Garder"),
    "bank_branch_count_8w": ("Demandes dans l'agence choisie sur 8 semaines", "Quantitative discrète", "Oui", "Garder"),
    "date_of_birth_distinct_emails_4w": ("Emails distincts pour la même date de naissance sur 4 semaines", "Quantitative discrète", "Oui", "Garder"),
    "employment_status": ("Statut d'emploi (anonymisé)", "Qualitative nominale", "Oui", "Garder — proxy social"),
    "credit_risk_score": ("Score de risque de crédit interne", "Quantitative", "À vérifier", "À discuter (fuite possible)"),
    "email_is_free": ("Domaine d'email gratuit", "Binaire", "Oui", "Garder"),
    "housing_status": ("Statut de logement (anonymisé)", "Qualitative nominale", "Oui", "Garder — proxy social"),
    "phone_home_valid": ("Téléphone fixe valide", "Binaire", "Oui", "Garder"),
    "phone_mobile_valid": ("Téléphone mobile valide", "Binaire", "Oui", "Garder"),
    "bank_months_count": ("Ancienneté du compte précédent (mois)", "Quantitative discrète", "Oui", "Garder"),
    "has_other_cards": ("Détient d'autres cartes dans la banque", "Binaire", "Oui", "Garder"),
    "proposed_credit_limit": ("Limite de crédit proposée", "Quantitative discrète", "À vérifier", "À discuter (fuite possible)"),
    "foreign_request": ("Demande depuis un pays différent de celui de la banque", "Binaire", "Oui", "Garder"),
    "source": ("Canal de la demande (INTERNET, TELEAPP)", "Qualitative nominale", "Oui", "Garder"),
    "session_length_in_minutes": ("Durée de la session (minutes)", "Quantitative continue", "Oui", "Garder"),
    "device_os": ("Système d'exploitation de l'appareil", "Qualitative nominale", "Oui", "Garder"),
    "keep_alive_session": ("Option « rester connecté » activée", "Binaire", "Oui", "Garder"),
    "device_distinct_emails_8w": ("Emails distincts depuis le même appareil sur 8 semaines", "Quantitative discrète", "Oui", "Garder"),
    "device_fraud_count": ("Demandes frauduleuses déjà faites depuis l'appareil", "Quantitative discrète", "À vérifier", "À discuter (fuite possible)"),
    "month": ("Mois de la demande (0 à 7)", "Temporelle", "Oui", "Découpage uniquement"),
}

rows = []
for var, (desc, typ, known, status) in DICTIONNAIRE.items():
    present = var in df.columns
    rows.append({
        "variable": var,
        "description": desc,
        "type": typ,
        "connue_au_depot": known,
        "statut": status if present else "Supprimée (constante)",
        "dtype_final": str(df[var].dtype) if present else None,
        "nb_manquants": int(df[var].isna().sum()) if present else None,
        "pct_manquants": round(100 * df[var].isna().mean(), 2) if present else None,
        "nb_valeurs_distinctes": int(df[var].nunique()) if present else None,
    })
for c in [c for c in df.columns if c.endswith("_manquant")] + ["split"]:
    rows.append({"variable": c, "description": "Colonne créée par le notebook", "type": "Binaire" if c != "split" else "Qualitative nominale",
                 "connue_au_depot": "Oui" if c != "split" else "Sans objet", "statut": "Créée", "dtype_final": str(df[c].dtype),
                 "nb_manquants": 0, "pct_manquants": 0.0, "nb_valeurs_distinctes": int(df[c].nunique())})

dictionnaire = pd.DataFrame(rows)
dictionnaire

,variable,description,type,connue_au_depot,statut,dtype_final,nb_manquants,pct_manquants,nb_valeurs_distinctes
0,fraud_bool,Demande frauduleuse (1) ou légitime (0),Binaire,Non (cible),Cible,int8,0.0,0.00,2.0
1,income,"Revenu déclaré, en déciles (0,1 à 0,9)",Quantitative ordonnée,Oui,Garder — sensible,float64,0.0,0.00,9.0
2,name_email_similarity,Similarité entre le nom et l'email (0 à 1),Quantitative continue,Oui,Garder,float64,0.0,0.00,998861.0
3,prev_address_months_count,Ancienneté à l'adresse précédente (mois),Quantitative discrète,Oui,Garder,float64,712920.0,71.29,373.0
4,current_address_months_count,Ancienneté à l'adresse actuelle (mois),Quantitative discrète,Oui,Garder,float64,4254.0,0.43,422.0
5,customer_age,Âge par tranches de 10 ans,Qualitative ordinale,Oui,Garder — sensible,int64,0.0,0.00,9.0
6,days_since_request,Délai depuis la création de la demande (jours),Quantitative continue,Oui,Garder,float64,0.0,0.00,989330.0
7,intended_balcon_amount,Montant initial prévu,Quantitative continue,Oui,Garder,float64,742523.0,74.25,257137.0
8,payment_type,Type de plan de paiement (anonymisé),Qualitative nominale,Oui,Garder,category,0.0,0.00,5.0
9,zip_count_4w,Demandes avec le même code postal sur 4 semaines,Quantitative discrète,Oui,Garder,int64,0.0,0.00,6306.0


## 8. Export et journal de lineage

In [42]:
clean_path = OUT_DIR / "base_clean.parquet"
dict_path = OUT_DIR / "dictionnaire_variables.csv"
quality_path = OUT_DIR / "rapport_qualite.json"
lineage_path = ROOT / "lineage.json"

df.to_parquet(clean_path, index=False)
dictionnaire.to_csv(dict_path, index=False, encoding="utf-8")
quality_path.write_text(json.dumps(quality, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

lineage = {
    "projet": "Atelier fil rouge M1 — Groupe D — fraude à l'ouverture de compte",
    "execution": {
        "date_execution": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "python": sys.version.split()[0],
        "pandas": pd.__version__,
        "systeme": platform.platform(),
    },
    "sources": [{
        "nom": "Bank Account Fraud Dataset Suite (Feedzai, NeurIPS 2022) — Base.csv",
        "url": SOURCE_URL,
        "identifiant_kaggle": KAGGLE_HANDLE,
        "documentation": [DOC_URL, DATASHEET_URL],
        "fichier": FILE_NAME,
        "methode_acces": download_meta["methode"],
        "version": download_meta["version"],
        "date_recuperation": download_meta["date_recuperation"],
        "sha256": raw_sha256,
        "taille_octets": raw_size,
        "licence": LICENCE,
        "nature": "Données générées à partir d'un jeu réel anonymisé (pas de données réelles)",
    }],
    "transformations": TRANSFORMATIONS,
    "sorties": [
        {"fichier": p.name, "sha256": sha256(p), "taille_octets": p.stat().st_size}
        for p in (clean_path, dict_path, quality_path)
    ],
}
lineage_path.write_text(json.dumps(lineage, ensure_ascii=False, indent=2), encoding="utf-8")

print(json.dumps(lineage, ensure_ascii=False, indent=2))

{
  "projet": "Atelier fil rouge M1 — Groupe D — fraude à l'ouverture de compte",
  "execution": {
    "date_execution": "2026-10-07T15:05:08+00:00",
    "python": "3.12.13",
    "pandas": "3.0.6",
    "systeme": "macOS-26.3.1-arm64-arm-64bit"
  },
  "sources": [
    {
      "nom": "Bank Account Fraud Dataset Suite (Feedzai, NeurIPS 2022) — Base.csv",
      "url": "https://www.kaggle.com/datasets/sgpjesus/bank-account-fraud-dataset-neurips2022",
      "identifiant_kaggle": "sgpjesus/bank-account-fraud-dataset-neurips2022",
      "documentation": [
        "https://github.com/feedzai/bank-account-fraud",
        "https://github.com/feedzai/bank-account-fraud"
      ],
      "fichier": "Base.csv",
      "methode_acces": "Téléchargement manuel depuis Kaggle",
      "version": "1",
      "date_recuperation": "2026-10-07T15:05:04+00:00",
      "sha256": "7bf10a37ce07e72e14c1b09e5efee3d27261baff4facc7da767b0474dcf9b809",
      "taille_octets": 213427735,
      "licence": "CC BY-NC-SA 4.0",
 